<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/8_15_26_Indigo_Random_Forest_YFinance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [2]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:00<00:00, 117MB/s]


In [3]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [4]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [7]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [6]:
len(tickers)

503

In [8]:
tickers[499:503]

['YUM', 'ZBRA', 'ZBH', 'ZTS']

In [11]:
import yfinance as yf
df = yf.download(tickers, period="3mo")
df.head()

/tmp/ipykernel_7177/88378548.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="3mo")
[*********************100%***********************]  503 of 503 completed


Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2026-05-15  111.485558  299.971313  208.903061  132.850006  83.871803   
2026-05-18  111.894775  297.583344  207.929993  134.300003  87.287445   
2026-05-19  110.357735  298.712372  212.249237  131.160004  88.190994   
2026-05-20  113.561569  301.989563  210.799561  135.550003  87.754105   
2026-05-21  114.569633  304.727173  212.984009  134.250000  87.148430   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2026-05-15  93.980003  166.814194  247.600006  416.349579  79.873383  ...   
2026-05-18  95.870003  175.440475  255.639999  417.436584  80.380043  ...   
2026-05-19  96.389999  174.699387  254.990005  413.178253  78.869995  ...   
2026-05-20  96.839996  177.090622  253.369995  396.962677  77.550003  ...   
2026-05-21  96.129997  175.756653  244.100006  383.160461  77.120003  ...   

Price        Volume                                                        \
Ticker           WY       WYNN        XEL       XOM        XYL        XYZ   
Date                                                                        
2026-05-15  6414400  1028100.0  4702200.0  27882400  2303900.0  5109300.0   
2026-05-18  4714800  1341000.0  4971700.0  20065200  2342400.0  4806000.0   
2026-05-19  5812800  1002200.0  3642800.0  20715600  1794300.0  5723800.0   
2026-05-20  5471200  1550400.0  4506200.0  18509700  1737900.0  6427100.0   
2026-05-21  4263800   904000.0  6243200.0  17061700  1547000.0  8496800.0   

Price                                                 
Ticker            YUM        ZBH       ZBRA      ZTS  
Date                                                  
2026-05-15  2156600.0  2578900.0  1224600.0  8952000  
2026-05-18  1418300.0  2303200.0  1067700.0  8848300  
2026-05-19  1625300.0  2918400.0  1096000.0  7295300  
2026-05-20  2501800.0  2569100.0  1102300.0  7404700  
2026-05-21  1891400.0  1895400.0  1013300.0  8564800  

[5 rows x 2515 columns]

In [12]:
closes = df["Close"]
returns = closes.pct_change() *100

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
fdf = pd.concat([cs,rs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)



/tmp/ipykernel_7177/3524405700.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return
501,2026-05-18,A,111.894775,0.367059
502,2026-05-18,AAPL,297.583344,-0.796066
503,2026-05-18,ABBV,207.929993,-0.465799
504,2026-05-18,ABNB,134.300003,1.091454
505,2026-05-18,ABT,87.287445,4.072455
506,2026-05-18,ACGL,95.870003,2.011065
507,2026-05-18,ACN,175.440475,5.171192
508,2026-05-18,ADBE,255.639999,3.247170
509,2026-05-18,ADI,417.436584,0.261080
510,2026-05-18,ADM,80.380043,0.634330


In [35]:
fdf.sort_values(by=['ticker', 'date'], inplace=True)
fdf.head(25)

,date,ticker,close,return
501,2026-05-18,A,111.894775,0.367059
1002,2026-05-19,A,110.357735,-1.373648
1503,2026-05-20,A,113.561569,2.903135
2004,2026-05-21,A,114.569633,0.887681
2505,2026-05-22,A,114.739304,0.148093
3006,2026-05-26,A,114.859077,0.104388
3507,2026-05-27,A,115.617615,0.660407
4009,2026-05-28,A,135.120117,16.868107
4511,2026-05-29,A,135.269821,0.110793
5013,2026-06-01,A,135.718948,0.332023


In [40]:
fdf['Lag 1'] = fdf['close'].shift(1)
fdf['Lag 2'] = fdf['close'].shift(2)
fdf['Lag 3'] = fdf['close'].shift(3)
fdf['Lag 5'] = fdf['close'].shift(5)
fdf.head(25)

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5
501,2026-05-18,A,111.894775,0.367059,NaN,NaN,NaN,NaN
1002,2026-05-19,A,110.357735,-1.373648,111.894775,NaN,NaN,NaN
1503,2026-05-20,A,113.561569,2.903135,110.357735,111.894775,NaN,NaN
2004,2026-05-21,A,114.569633,0.887681,113.561569,110.357735,111.894775,NaN
2505,2026-05-22,A,114.739304,0.148093,114.569633,113.561569,110.357735,NaN
3006,2026-05-26,A,114.859077,0.104388,114.739304,114.569633,113.561569,111.894775
3507,2026-05-27,A,115.617615,0.660407,114.859077,114.739304,114.569633,110.357735
4009,2026-05-28,A,135.120117,16.868107,115.617615,114.859077,114.739304,113.561569
4511,2026-05-29,A,135.269821,0.110793,135.120117,115.617615,114.859077,114.569633
5013,2026-06-01,A,135.718948,0.332023,135.269821,135.120117,115.617615,114.739304


In [56]:
fdf['SMA 5'] = fdf['close'].rolling(window=5).mean()
fdf['SMA 10'] = fdf['close'].rolling(window=10).mean()
fdf['SMA 20'] = fdf['close'].rolling(window=20).mean()
fdf['SMA 30'] = fdf['close'].rolling(window=30).mean()
fdf['SMA 50'] = fdf['close'].rolling(window=50).mean()

fdf.head(100)
fdf_appl = fdf[fdf['ticker'] == 'AAPL'].copy()
fdf_appl.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,SMA 100
502,2026-05-18,AAPL,297.583344,-0.796066,148.529999,149.289993,148.240005,148.720001,178.626669,160.947334,149.485668,143.985779,138.742865,NaN
1003,2026-05-19,AAPL,298.712372,0.379399,297.583344,148.529999,149.289993,149.490005,208.471143,176.898572,157.778287,149.571525,142.013513,NaN
1504,2026-05-20,AAPL,301.989563,1.097106,298.712372,297.583344,148.529999,148.240005,239.221054,192.986528,166.204765,155.335510,145.404598,NaN
2005,2026-05-21,AAPL,304.727173,0.906525,301.989563,298.712372,297.583344,149.289993,270.308490,209.325246,174.454123,161.040082,148.794744,NaN
2506,2026-05-22,AAPL,308.553894,1.255786,304.727173,301.989563,298.712372,148.529999,302.313269,225.583635,182.953317,166.848879,152.338475,NaN


In [57]:
len(fdf_appl)
fdf_appl.drop(columns = 'SMA 100')

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50
502,2026-05-18,AAPL,297.583344,-0.796066,148.529999,149.289993,148.240005,148.720001,178.626669,160.947334,149.485668,143.985779,138.742865
1003,2026-05-19,AAPL,298.712372,0.379399,297.583344,148.529999,149.289993,149.490005,208.471143,176.898572,157.778287,149.571525,142.013513
1504,2026-05-20,AAPL,301.989563,1.097106,298.712372,297.583344,148.529999,148.240005,239.221054,192.986528,166.204765,155.335510,145.404598
2005,2026-05-21,AAPL,304.727173,0.906525,301.989563,298.712372,297.583344,149.289993,270.308490,209.325246,174.454123,161.040082,148.794744
2506,2026-05-22,AAPL,308.553894,1.255786,304.727173,301.989563,298.712372,148.529999,302.313269,225.583635,182.953317,166.848879,152.338475
...,...,...,...,...,...,...,...,...,...,...,...,...,...
28709,2026-08-10,AAPL,308.260010,-1.533249,313.059998,312.140808,310.732025,303.158569,310.661249,317.593683,322.713972,317.043765,309.438845
29211,2026-08-11,AAPL,304.910004,-1.086747,308.260010,313.059998,312.140808,309.113403,309.820569,314.105991,322.230038,317.824191,309.301223
29714,2026-08-12,AAPL,302.250000,-0.872390,304.910004,308.260010,313.059998,310.732025,308.124164,310.541132,320.981648,318.262169,309.225302
30217,2026-08-13,AAPL,305.260010,0.995868,302.250000,304.910004,308.260010,312.140808,306.748004,307.752866,319.596007,318.633292,309.031934


In [58]:
fdf_appl['volatility 5'] = fdf_appl['return'].rolling(5).std()
fdf_appl['volatility 10'] = fdf_appl['return'].rolling(10).std()
fdf_appl['volatility 20'] = fdf_appl['return'].rolling(20).std()
fdf_appl['volatility 30'] = fdf_appl['return'].rolling(30).std()
fdf_appl.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,SMA 100,volatility 5,volatility 10,volatility 20,volatility 30
502,2026-05-18,AAPL,297.583344,-0.796066,148.529999,149.289993,148.240005,148.720001,178.626669,160.947334,149.485668,143.985779,138.742865,NaN,NaN,NaN,NaN,NaN
1003,2026-05-19,AAPL,298.712372,0.379399,297.583344,148.529999,149.289993,149.490005,208.471143,176.898572,157.778287,149.571525,142.013513,NaN,NaN,NaN,NaN,NaN
1504,2026-05-20,AAPL,301.989563,1.097106,298.712372,297.583344,148.529999,148.240005,239.221054,192.986528,166.204765,155.335510,145.404598,NaN,NaN,NaN,NaN,NaN
2005,2026-05-21,AAPL,304.727173,0.906525,301.989563,298.712372,297.583344,149.289993,270.308490,209.325246,174.454123,161.040082,148.794744,NaN,NaN,NaN,NaN,NaN
2506,2026-05-22,AAPL,308.553894,1.255786,304.727173,301.989563,298.712372,148.529999,302.313269,225.583635,182.953317,166.848879,152.338475,NaN,0.831241,NaN,NaN,NaN


In [60]:
fdf_appl = fdf_appl.dropna()
fdf_appl = fdf_appl.drop(columns = 'SMA 100')
fdf_appl.head()


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
19088,2026-07-13,AAPL,317.036560,0.631094,315.048309,315.947510,313.119965,312.390594,314.308929,305.703351,298.929192,300.588760,263.777727,0.712912,1.685220,2.202867,2.115220
19591,2026-07-14,AAPL,314.588684,-0.772112,317.036560,315.048309,315.947510,310.392303,315.148206,309.012497,300.114668,300.682013,267.295300,0.757511,1.691802,2.175029,2.120540
20094,2026-07-15,AAPL,327.217804,4.014486,314.588684,317.036560,315.048309,313.119965,317.967773,312.823212,301.667329,301.387738,271.072256,1.868929,1.868700,2.301399,2.207951
20597,2026-07-16,AAPL,332.972839,1.758778,327.217804,314.588684,317.036560,315.947510,321.372839,316.707861,303.366864,301.989219,274.935713,1.908231,1.869388,2.316303,2.170074
21100,2026-07-17,AAPL,333.452393,0.144022,332.972839,327.217804,314.588684,315.048309,325.053656,319.216696,305.254735,302.771211,278.820761,1.840620,1.402532,2.285748,2.144116


len(fdf_appl)


In [63]:
fdf_appl.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
19088,2026-07-13,AAPL,317.036560,0.631094,315.048309,315.947510,313.119965,312.390594,314.308929,305.703351,298.929192,300.588760,263.777727,0.712912,1.685220,2.202867,2.115220
19591,2026-07-14,AAPL,314.588684,-0.772112,317.036560,315.048309,315.947510,310.392303,315.148206,309.012497,300.114668,300.682013,267.295300,0.757511,1.691802,2.175029,2.120540
20094,2026-07-15,AAPL,327.217804,4.014486,314.588684,317.036560,315.048309,313.119965,317.967773,312.823212,301.667329,301.387738,271.072256,1.868929,1.868700,2.301399,2.207951
20597,2026-07-16,AAPL,332.972839,1.758778,327.217804,314.588684,317.036560,315.947510,321.372839,316.707861,303.366864,301.989219,274.935713,1.908231,1.869388,2.316303,2.170074
21100,2026-07-17,AAPL,333.452393,0.144022,332.972839,327.217804,314.588684,315.048309,325.053656,319.216696,305.254735,302.771211,278.820761,1.840620,1.402532,2.285748,2.144116


In [71]:
fdf_appl['future return'] = fdf_appl.groupby('ticker')['return'].shift(-1)
fdf_appl = fdf_appl.dropna()
print(len(fdf_appl))
fdf_appl.head(len(fdf_appl))


24


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,future return
19088,2026-07-13,AAPL,317.036560,0.631094,315.048309,315.947510,313.119965,312.390594,314.308929,305.703351,298.929192,300.588760,263.777727,0.712912,1.685220,2.202867,2.115220,-0.772112
19591,2026-07-14,AAPL,314.588684,-0.772112,317.036560,315.048309,315.947510,310.392303,315.148206,309.012497,300.114668,300.682013,267.295300,0.757511,1.691802,2.175029,2.120540,4.014486
20094,2026-07-15,AAPL,327.217804,4.014486,314.588684,317.036560,315.048309,313.119965,317.967773,312.823212,301.667329,301.387738,271.072256,1.868929,1.868700,2.301399,2.207951,1.758778
20597,2026-07-16,AAPL,332.972839,1.758778,327.217804,314.588684,317.036560,315.947510,321.372839,316.707861,303.366864,301.989219,274.935713,1.908231,1.869388,2.316303,2.170074,0.144022
21100,2026-07-17,AAPL,333.452393,0.144022,332.972839,327.217804,314.588684,315.048309,325.053656,319.216696,305.254735,302.771211,278.820761,1.840620,1.402532,2.285748,2.144116,-2.142384
21603,2026-07-20,AAPL,326.308563,-2.142384,333.452393,332.972839,327.217804,317.036560,326.908057,320.608493,306.682503,303.282769,282.524732,2.377644,1.662495,2.367920,2.188655,0.352126
22106,2026-07-21,AAPL,327.457581,0.352126,326.308563,333.452393,332.972839,314.588684,329.481836,322.315021,308.217677,303.962183,286.247084,2.265914,1.619346,2.360347,2.171944,-0.564467
22445,2026-07-22,AAPL,325.609192,-0.564467,327.457581,326.308563,333.452393,327.217804,329.160114,323.563943,309.795818,304.773150,289.839868,1.423518,1.650634,2.350494,2.140527,-1.297982
22773,2026-07-23,AAPL,321.382843,-1.297982,325.609192,327.457581,326.308563,332.972839,326.842114,324.107477,311.223589,305.809257,293.293125,1.033894,1.721904,2.377473,2.032246,3.531672
23276,2026-07-24,AAPL,332.733032,3.531672,321.382843,325.609192,327.457581,333.452393,326.698242,325.875949,314.114595,307.189400,296.957985,2.190146,2.005999,1.895482,2.113141,1.168101


In [72]:
features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50']
x = fdf_appl[features]
y = fdf_appl['future return']



In [77]:
split_index = int(0.8 * len(fdf_appl))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [78]:
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor
tree = DecisionTreeRegressor(max_depth = 5, random_state=72)

tree.fit(x_train, y_train)
predictions = tree.predict(x_test)

score = mean_squared_error(y_test, predictions)
score


10.280040448509299

In [80]:
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor(n_estimators = 500, random_state=72)
forest.fit(x_train, y_train)

RandomForestRegressor(n_estimators=500, random_state=72)

In [81]:
predictions = forest.predict(x_test)
score = mean_squared_error(y_test, predictions)
score

5.860034035667014